In [1]:
import sys
sys.path.append("../") 
from libs.cna_utils import *
# For reproducibility
np.random.seed(0) 

# RESPONSE

## Single Cell

In [2]:
meta = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/age_sex_response_race/sc_meta_response.csv')
print(meta.shape)

(27974, 37)


In [3]:
harmony = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/age_sex_response_race/sc_harmony_response.csv')
print(harmony.shape)

(27974, 20)


In [4]:
umap = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/age_sex_response_race/sc_umap_response.csv')
print(umap.shape)

(27974, 2)


In [5]:
# Start by making anndata object
d = mad.MultiAnnData(X=harmony, obs=meta, sampleid="sample")
#d.obs_to_sample(['Type', 'Site',
#                 'Responder_Status', 'Sex',
#                 'Age', 'Race',
#                'Ethnicity', 'ISN',
#               'Activity', "Chronicity",
#                 'processing_batch'])
d.obs_to_sample(['Sex', 'Age', 'Final_Chronicity', 'Final_Activity', 'First_biop', 'Pred_use', 'Responder_Status', 'Race_[A]',
       'Race_[A][B]', 'Race_[B]', 'Race_[B][AI]', 'Race_[U]',
       'Race_[W]', 'Final_ISN_[III]', 'Final_ISN_[III][V]', 'Final_ISN_[IV]',
       'Final_ISN_[IV][V]', 'Final_ISN_[V]', 
       'Final_Site_Einstein', 'Final_Site_JHU', 'Final_Site_Michigan',
       'Final_Site_MUSC', 'Final_Site_Northwell', 'Final_Site_NYU',
       'Final_Site_Rochester', 'Final_Site_Texas Tech', 
       'Final_Site_UCSD', 'Final_Site_UCSF', 'Responder_Status', 'injured_pt_prop'])
d.samplem.head()

['cell' 'sample' 'final_annotation' 'Responder.Status' 'Race' 'Final_ISN'
 'Type' 'Final_Site']
consider casting to numeric types where appropriate, and
consider re-coding text-valued columns with pandas.get_dummies


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/multianndata/core.py:17: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  super().__init__(*args, **kwargs)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


,Sex,Age,Final_Chronicity,Final_Activity,First_biop,Pred_use,Responder_Status,Race_[A],Race_[A][B],Race_[B],...,Final_Site_JHU,Final_Site_Michigan,Final_Site_MUSC,Final_Site_Northwell,Final_Site_NYU,Final_Site_Rochester,Final_Site_Texas Tech,Final_Site_UCSD,Final_Site_UCSF,injured_pt_prop
sample,,,,,,,,,,,,,,,,,,,,,
AMPSLEkid_cells_0134,1.0,0.849071,6.0,4.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,16.551724
AMPSLEkid_cells_0137,1.0,-0.129795,3.0,5.0,0.0,1.0,2.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,14.341498
AMPSLEkid_cells_0138,1.0,0.404132,0.0,0.0,1.0,1.0,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,5.629663
AMPSLEkid_cells_0139,1.0,0.760083,3.0,2.0,0.0,1.0,2.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,4.405520
AMPSLEkid_cells_0140,1.0,-0.930685,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,6.399412


In [6]:
umap.index = d.obs.index
d.obsm['X_umap'] = umap

In [7]:
np.random.seed(0) 
cna.pp.knn(d)

computing default knn graph


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
np.random.seed(0) 
res = cna.tl._association.association(d, #dataset 
                                      d.samplem.Responder_Status, #phenotype
#                                       batches=d.samplem.processing_batch, #batches
                                      Nnull=10000, # number of null permutations to do (defaults to only 1e3)
                                      ks=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] # I asked the method to consider up to 10 PCs because
                                                                #it chose the max number of PCs it considered the default set of [1,2,3,4]
                                     )
print(res.k)
print(res.ks)
print('p =', res.p, ',', res.k, 'PCs used')
print('total r^2 between top {} NAM PCs and outcome is {:.2f}'.format(res.k, res.r2))

qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 41.67474365234375
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 26.450233459472656
	20th percentile R2(t,t-1): 0.746048092842102
	taking step 3
	median kurtosis: 18.167381286621094
	20th percentile R2(t,t-1): 0.8998205661773682
	taking step 4
	median kurtosis: 12.966375350952148
	20th percentile R2(t,t-1): 0.940981924533844
	taking step 5
	median kurtosis: 10.036448955535889
	20th percentile R2(t,t-1): 0.9636072516441345
stopping after 5 steps
covariate-adjusted NAM not found; computing and saving


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


computing SVD
performing association test
computing neighborhood-level FDRs
12
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.0047995200479952005 , 12 PCs used
total r^2 between top 12 NAM PCs and outcome is 0.25


In [9]:
response_uni = res.p

In [10]:
np.random.seed(0) 
res = cna.tl._association.association(d, #dataset 
                                      d.samplem.Responder_Status, #phenotype
#                                       batches=d.samplem.processing_batch, #batches
                                      covs = d.samplem[['Final_Chronicity']],
                                      Nnull=10000, # number of null permutations to do (defaults to only 1e3)
                                      ks=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] # I asked the method to consider up to 10 PCs because
                                                                #it chose the max number of PCs it considered the default set of [1,2,3,4]
                                     )
print(res.k)
print(res.ks)
print('p =', res.p, ',', res.k, 'PCs used')
print('total r^2 between top {} NAM PCs and outcome is {:.2f}'.format(res.k, res.r2))

covariate-adjusted NAM not found; computing and saving
computing SVD
performing association test


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


computing neighborhood-level FDRs
12
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.21547845215478453 , 12 PCs used
total r^2 between top 12 NAM PCs and outcome is 0.18


In [11]:
response_cond = res.p

In [59]:
sc_uni['Responder_Status'] = res.p

In [60]:
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/age_sex_response_race/sc_response_ncorr.csv", 
               res.ncorrs, delimiter=",")
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/age_sex_response_race/sc_response_fdrs.csv", 
               res.fdrs, delimiter=",")

In [39]:
np.random.seed(0) 
res = cna.tl._association.association(d, #dataset 
                                      d.samplem.Responder_Status, #phenotype
#                                       batches=d.samplem.processing_batch, #batches
                                      covs = d.samplem[['Final_Chronicity', 'First_biop']],
                                      Nnull=10000, # number of null permutations to do (defaults to only 1e3)
                                      ks=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] # I asked the method to consider up to 10 PCs because
                                                                #it chose the max number of PCs it considered the default set of [1,2,3,4]
                                     )
print(res.k)
print(res.ks)
print('p =', res.p, ',', res.k, 'PCs used')
print('total r^2 between top {} NAM PCs and outcome is {:.2f}'.format(res.k, res.r2))

qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 41.67474365234375
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 26.450233459472656
	20th percentile R2(t,t-1): 0.746048092842102
	taking step 3
	median kurtosis: 18.167381286621094
	20th percentile R2(t,t-1): 0.8998205661773682
	taking step 4
	median kurtosis: 12.966375350952148
	20th percentile R2(t,t-1): 0.940981924533844
	taking step 5
	median kurtosis: 10.036448955535889
	20th percentile R2(t,t-1): 0.9636072516441345
stopping after 5 steps
covariate-adjusted NAM not found; computing and saving
computing SVD


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


performing association test
computing neighborhood-level FDRs
12
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.1146885311468853 , 12 PCs used
total r^2 between top 12 NAM PCs and outcome is 0.20


In [40]:
sc_cond['Responder_Status'] = res.p

In [13]:
np.random.seed(0) 
res = cna.tl._association.association(d, #dataset 
                                      d.samplem.Responder_Status, #phenotype
#                                       batches=d.samplem.processing_batch, #batches
                                      covs = d.samplem[['injured_pt_prop']], 
                                      Nnull=10000, # number of null permutations to do (defaults to only 1e3)
                                      ks=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] # I asked the method to consider up to 10 PCs because
                                                                #it chose the max number of PCs it considered the default set of [1,2,3,4]
                                     )
print(res.k)
print(res.ks)
print('p =', res.p, ',', res.k, 'PCs used')
print('total r^2 between top {} NAM PCs and outcome is {:.2f}'.format(res.k, res.r2))

qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 41.67474365234375
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 26.450233459472656
	20th percentile R2(t,t-1): 0.746048092842102
	taking step 3
	median kurtosis: 18.167381286621094
	20th percentile R2(t,t-1): 0.8998205661773682
	taking step 4
	median kurtosis: 12.966375350952148
	20th percentile R2(t,t-1): 0.940981924533844
	taking step 5
	median kurtosis: 10.036448955535889
	20th percentile R2(t,t-1): 0.9636072516441345
stopping after 5 steps
covariate-adjusted NAM not found; computing and saving


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


computing SVD
performing association test
computing neighborhood-level FDRs
11
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.027997200279972004 , 11 PCs used
total r^2 between top 11 NAM PCs and outcome is 0.21


In [16]:
np.random.seed(0) 
res = cna.tl._association.association(d, #dataset 
                                      d.samplem.injured_pt_prop, #phenotype
#                                       batches=d.samplem.processing_batch, #batches
                                      covs = d.samplem[['Responder_Status']], 
                                      Nnull=10000, # number of null permutations to do (defaults to only 1e3)
                                      ks=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] # I asked the method to consider up to 10 PCs because
                                                                #it chose the max number of PCs it considered the default set of [1,2,3,4]
                                     )
print(res.k)
print(res.ks)
print('p =', res.p, ',', res.k, 'PCs used')
print('total r^2 between top {} NAM PCs and outcome is {:.2f}'.format(res.k, res.r2))

covariate-adjusted NAM not found; computing and saving
computing SVD
performing association test


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


computing neighborhood-level FDRs
13
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.0008999100089991 , 13 PCs used
total r^2 between top 13 NAM PCs and outcome is 0.31


In [15]:
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/age_sex_response_race/sc_injured_pt_ncorr.csv", 
               res.ncorrs, delimiter=",")
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/age_sex_response_race/sc_injured_pt_fdrs.csv", 
               res.fdrs, delimiter=",")

In [12]:
np.random.seed(0) 
res = cna.tl._association.association(d, #dataset 
                                      d.samplem.Final_Chronicity, #phenotype
#                                       batches=d.samplem.processing_batch, #batches
                                      Nnull=10000, # number of null permutations to do (defaults to only 1e3)
                                      ks=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] # I asked the method to consider up to 10 PCs because
                                                                #it chose the max number of PCs it considered the default set of [1,2,3,4]
                                     )
print(res.k)
print(res.ks)
print('p =', res.p, ',', res.k, 'PCs used')
print('total r^2 between top {} NAM PCs and outcome is {:.2f}'.format(res.k, res.r2))

covariate-adjusted NAM not found; computing and saving
computing SVD
performing association test


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_association.py:74: UserWarning: global association p-value attained minimal possible value. Consider increasing Nnull
  warnings.warn('global association p-value attained minimal possible value. '+\


computing neighborhood-level FDRs
11
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 9.999000099990002e-05 , 11 PCs used
total r^2 between top 11 NAM PCs and outcome is 0.35


In [13]:
chron_uni = res.p

In [14]:
np.random.seed(0) 
res = cna.tl._association.association(d, #dataset 
                                      d.samplem.Final_Chronicity, #phenotype
#                                       batches=d.samplem.processing_batch, #batches
                                      covs = d.samplem[['Responder_Status']],
                                      Nnull=10000, # number of null permutations to do (defaults to only 1e3)
                                      ks=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] # I asked the method to consider up to 10 PCs because
                                                                #it chose the max number of PCs it considered the default set of [1,2,3,4]
                                     )
print(res.k)
print(res.ks)
print('p =', res.p, ',', res.k, 'PCs used')
print('total r^2 between top {} NAM PCs and outcome is {:.2f}'.format(res.k, res.r2))

covariate-adjusted NAM not found; computing and saving
computing SVD
performing association test


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


computing neighborhood-level FDRs
11
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.000999900009999 , 11 PCs used
total r^2 between top 11 NAM PCs and outcome is 0.30


In [15]:
chron_cond = res.p

In [16]:
pd.DataFrame(dict(pval = [response_uni, response_cond, chron_uni, chron_cond],
             variable = ['Responder Status', 'Responder Status', 'Chronicity', 'Chronicity'],
             test = ['Univariate', 'Conditional', 'Univariate', 'Conditional'],
             cell_type = ['T/NK', 'T/NK', 'T/NK', 'T/NK'])).to_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/age_sex_response_race/response_chronicity_cond.csv')

## Single Nuclei

In [99]:
meta = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/age_sex_response_race/sn_meta_response.csv')
print(meta.shape)

(2394, 10)


In [100]:
harmony = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/age_sex_response_race/sn_harmony_response.csv')
print(harmony.shape)

(2394, 20)


In [101]:
umap = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/age_sex_response_race/sn_umap_response.csv')
print(umap.shape)

(2394, 2)


In [102]:
# Start by making anndata object
d = mad.MultiAnnData(X=harmony, obs=meta, sampleid="sample")
#d.obs_to_sample(['Type', 'Site',
#                 'Responder_Status', 'Sex',
#                 'Age', 'Race',
#                'Ethnicity', 'ISN',
#               'Activity', "Chronicity",
#                 'processing_batch'])
d.obs_to_sample(['Responder_Status'])
d.samplem.head()

['cell' 'Sex' 'sample' 'final_annotation' 'Responder.Status' 'Race' 'ISN'
 'Type']
consider casting to numeric types where appropriate, and
consider re-coding text-valued columns with pandas.get_dummies


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/multianndata/core.py:17: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  super().__init__(*args, **kwargs)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


,Responder_Status
sample,
AMPSLEkid_cells_0137,2.0
AMPSLEkid_cells_0138,2.0
AMPSLEkid_cells_0139,2.0
AMPSLEkid_cells_0147,1.0
AMPSLEkid_cells_0366,0.0


In [103]:
umap.index = d.obs.index
d.obsm['X_umap'] = umap

In [104]:
np.random.seed(0) 
cna.pp.knn(d)

computing default knn graph


In [105]:
np.random.seed(0) 
res = cna.tl._association.association(d, #dataset 
                                      d.samplem.Responder_Status, #phenotype
#                                       batches=d.samplem.processing_batch, #batches
                                      Nnull=10000, # number of null permutations to do (defaults to only 1e3)
                                      ks=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] # I asked the method to consider up to 10 PCs because
                                                                #it chose the max number of PCs it considered the default set of [1,2,3,4]
                                     )
print(res.k)
print(res.ks)
print('p =', res.p, ',', res.k, 'PCs used')
print('total r^2 between top {} NAM PCs and outcome is {:.2f}'.format(res.k, res.r2))

qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 9.336187625789051
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 5.902152485751815
	20th percentile R2(t,t-1): 0.7219850659370423
	taking step 3
	median kurtosis: 4.588057901388883
	20th percentile R2(t,t-1): 0.9186734914779663
stopping after 3 steps
covariate-adjusted NAM not found; computing and saving
computing SVD
performing association test


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


computing neighborhood-level FDRs
12
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.675032496750325 , 12 PCs used
total r^2 between top 12 NAM PCs and outcome is 0.39


In [106]:
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/age_sex_response_race/sn_response_ncorr.csv", 
               res.ncorrs, delimiter=",")
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/age_sex_response_race/sn_response_fdrs.csv", 
               res.fdrs, delimiter=",")

# CASE/CONTROL

## Single Cell

In [2]:
meta = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/case_control/sc_meta.csv')
print(meta.shape)

(35131, 78)


/tmp/ipykernel_49980/3380040273.py:1: DtypeWarning: Columns (59) have mixed types. Specify dtype option on import or set low_memory=False.
  meta = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/case_control/sc_meta.csv')


In [3]:
harmony = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/case_control/sc_harmony.csv')
print(harmony.shape)

(35131, 20)


In [4]:
umap = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/case_control/sc_umap.csv')
print(umap.shape)

(35131, 2)


In [5]:
res = cna_test(meta, harmony, umap, 'Type_numeric')

(35131, 78)
(35131, 20)
(35131, 2)
['cell' 'sample' 'Annot.separate' 'dataset' 'Site' 'broad.type'
 'doublet_classification' 'arnon_annotations' 'annotation'
 'final_annotation' 'annotation.order' 'kid_sample' 'AMP.ID' 'Final_Site'
 'Clinical.Response.at.12.weeks' 'Clinical.Response.at.26.weeks'
 'Clinical.Response.at.52.weeks' 'Responder.Status' 'Type' 'Sex' 'Race'
 'Ethnicity' 'ISN' 'processing.date' 'XTR.WTA.batch' 'DASH.batch' 'index'
 'seq.submission.date' 'Flow.cell' 'Rituximab.stop.date'
 'biopsy.length.from.site.mm' 'AMP.Subject_ID' 'Central_ISN' 'Final_ISN']
consider casting to numeric types where appropriate, and
consider re-coding text-valued columns with pandas.get_dummies
computing default knn graph


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/multianndata/core.py:17: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  super().__init__(*args, **kwargs)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 66.03807067871094
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 42.36387252807617
	20th percentile R2(t,t-1): 0.7422828674316406
	taking step 3
	median kurtosis: 29.72708511352539
	20th percentile R2(t,t-1): 0.8985448956489563
	taking step 4
	median kurtosis: 21.421955108642578
	20th percentile R2(t,t-1): 0.9385355114936829
	taking step 5
	median kurtosis: 16.20050621032715
	20th percentile R2(t,t-1): 0.9598562240600585
	taking step 6
	median kurtosis: 13.17811107635498
	20th percentile R2(t,t-1): 0.9743244886398316
	taking step 7
	median kurtosis: 11.303744316101074
	20th percentile R2(t,t-1): 0.9841291069984436
stopping after 7 steps
covariate-adjusted NAM not found; computing and saving
computing SVD


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


performing association test


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_association.py:74: UserWarning: global association p-value attained minimal possible value. Consider increasing Nnull
  warnings.warn('global association p-value attained minimal possible value. '+\


computing neighborhood-level FDRs
19
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 9.999000099990002e-05 , 19 PCs used
total r^2 between top 19 NAM PCs and outcome is 0.61


In [7]:
np.savetxt('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/case_control/sc_ncorr.csv', res.ncorrs, delimiter=",")
np.savetxt('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/case_control/sc_fdrs.csv', res.fdrs, delimiter=",")

# Chronicity

## Single Cell

In [7]:
meta = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/chronicity/sc_meta.csv')
print(meta.shape)

(31561, 39)


In [8]:
harmony = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/chronicity/sc_harmony.csv')
print(harmony.shape)

(31561, 20)


In [9]:
umap = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/chronicity/sc_umap.csv')
print(umap.shape)

(31561, 2)


In [10]:
res = cna_test(meta, harmony, umap, 'Final_Chronicity')

(31561, 39)
(31561, 20)
(31561, 2)
['cell' 'sample' 'final_annotation' 'Responder.Status' 'Race' 'Final_ISN'
 'Type' 'Final_Site']
consider casting to numeric types where appropriate, and
consider re-coding text-valued columns with pandas.get_dummies
computing default knn graph


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/multianndata/core.py:17: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  super().__init__(*args, **kwargs)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 47.986328125
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 30.828720092773438
	20th percentile R2(t,t-1): 0.7443044185638428
	taking step 3
	median kurtosis: 21.22922706604004
	20th percentile R2(t,t-1): 0.8989471077919007
	taking step 4
	median kurtosis: 15.12545108795166
	20th percentile R2(t,t-1): 0.9395679831504822
	taking step 5
	median kurtosis: 11.710641860961914
	20th percentile R2(t,t-1): 0.9621577382087707
	taking step 6
	median kurtosis: 9.634602546691895
	20th percentile R2(t,t-1): 0.9768100500106811
stopping after 6 steps
covariate-adjusted NAM not found; computing and saving


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


computing SVD
performing association test
computing neighborhood-level FDRs
12
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.00019998000199980003 , 12 PCs used
total r^2 between top 12 NAM PCs and outcome is 0.34


In [11]:
res = cna_test(meta,harmony, umap, 'Final_Chronicity', covars = ['First_biop', 'Responder_Status'])

(31561, 39)
(31561, 20)
(31561, 2)
['cell' 'sample' 'final_annotation' 'Responder.Status' 'Race' 'Final_ISN'
 'Type' 'Final_Site']
consider casting to numeric types where appropriate, and
consider re-coding text-valued columns with pandas.get_dummies
computing default knn graph


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/multianndata/core.py:17: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  super().__init__(*args, **kwargs)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 47.986328125
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 30.828720092773438
	20th percentile R2(t,t-1): 0.7443044185638428
	taking step 3
	median kurtosis: 21.22922706604004
	20th percentile R2(t,t-1): 0.8989471077919007
	taking step 4
	median kurtosis: 15.12545108795166
	20th percentile R2(t,t-1): 0.9395679831504822
	taking step 5
	median kurtosis: 11.710641860961914
	20th percentile R2(t,t-1): 0.9621577382087707
	taking step 6
	median kurtosis: 9.634602546691895
	20th percentile R2(t,t-1): 0.9768100500106811
stopping after 6 steps
covariate-adjusted NAM not found; computing and saving
computing SVD


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


performing association test
computing neighborhood-level FDRs
11
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.0057994200579942 , 11 PCs used
total r^2 between top 11 NAM PCs and outcome is 0.27


In [12]:
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/chronicity/sc_conditional_ncorr.csv", 
               res.ncorrs, delimiter=",")
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/chronicity/sc_conditional_fdrs.csv", 
               res.fdrs, delimiter=",")

## Single Nuclei

In [13]:
meta = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/chronicity/sn_meta.csv')
print(meta.shape)

(2120, 32)


In [14]:
harmony = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/chronicity/sn_harmony.csv')
print(harmony.shape)

(2120, 20)


In [15]:
umap = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/chronicity/sn_umap.csv')
print(umap.shape)

(2120, 2)


In [17]:
res = cna_test(meta, harmony, umap, 'Final_Chronicity')

(2120, 32)
(2120, 20)
(2120, 2)
['cell' 'sample' 'final_annotation' 'Responder.Status' 'Race' 'Final_ISN'
 'Type' 'Final_Site']
consider casting to numeric types where appropriate, and
consider re-coding text-valued columns with pandas.get_dummies
computing default knn graph


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/multianndata/core.py:17: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  super().__init__(*args, **kwargs)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 8.153996863617243
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 5.191700621962467
	20th percentile R2(t,t-1): 0.729617440700531
	taking step 3
	median kurtosis: 4.031588959218854
	20th percentile R2(t,t-1): 0.9231990098953247
stopping after 3 steps
covariate-adjusted NAM not found; computing and saving
computing SVD
performing association test


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


computing neighborhood-level FDRs
1
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.2703729627037296 , 1 PCs used
total r^2 between top 1 NAM PCs and outcome is 0.20


In [ ]:
res = cna_test(meta,harmony, umap, 'Final_Chronicity', covars = ['First_biop', 'Responder_Status'])

In [160]:
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/chronicity/sn_ncorr.csv", 
               res.ncorrs, delimiter=",")
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/chronicity/sn_fdrs.csv", 
               res.fdrs, delimiter=",")

# Activity

## Single Cell

In [103]:
meta = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/activity/sc_meta.csv')
print(meta.shape)

(32485, 39)


In [104]:
harmony = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/activity/sc_harmony.csv')
print(harmony.shape)

(32485, 20)


In [105]:
umap = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/activity/sc_umap.csv')
print(umap.shape)

(32485, 2)


In [106]:
# Start by making anndata object
d = mad.MultiAnnData(X=harmony, obs=meta, sampleid="sample")
#d.obs_to_sample(['Type', 'Site',
#                 'Responder_Status', 'Sex',
#                 'Age', 'Race',
#                'Ethnicity', 'ISN',
#               'Activity', "Chronicity",
#                 'processing_batch'])
d.obs_to_sample(['Final_Activity'])
d.samplem.head()

['cell' 'sample' 'final_annotation' 'Responder.Status' 'Race' 'Final_ISN'
 'Type' 'Final_Site']
consider casting to numeric types where appropriate, and
consider re-coding text-valued columns with pandas.get_dummies


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/multianndata/core.py:17: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  super().__init__(*args, **kwargs)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


,Final_Activity
sample,
AMPSLEkid_cells_0134,4.0
AMPSLEkid_cells_0137,5.0
AMPSLEkid_cells_0138,0.0
AMPSLEkid_cells_0139,2.0
AMPSLEkid_cells_0140,1.0


In [107]:
umap.index = d.obs.index
d.obsm['X_umap'] = umap

In [108]:
np.random.seed(0) 
cna.pp.knn(d)

computing default knn graph


In [109]:
np.random.seed(0) 
res = cna.tl._association.association(d, #dataset 
                                      d.samplem.Final_Activity, #phenotype
#                                       batches=d.samplem.processing_batch, #batches
                                      Nnull=10000, # number of null permutations to do (defaults to only 1e3)
                                      ks=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] # I asked the method to consider up to 10 PCs because
                                                                #it chose the max number of PCs it considered the default set of [1,2,3,4]
                                     )
print(res.k)
print(res.ks)
print('p =', res.p, ',', res.k, 'PCs used')
print('total r^2 between top {} NAM PCs and outcome is {:.2f}'.format(res.k, res.r2))

qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 48.137882232666016
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 30.84671974182129
	20th percentile R2(t,t-1): 0.7438087105751038
	taking step 3
	median kurtosis: 21.176321029663086
	20th percentile R2(t,t-1): 0.8987076044082641
	taking step 4
	median kurtosis: 15.017460823059082
	20th percentile R2(t,t-1): 0.9393186211585999
	taking step 5
	median kurtosis: 11.646937370300293
	20th percentile R2(t,t-1): 0.961981725692749
	taking step 6
	median kurtosis: 9.655332565307617
	20th percentile R2(t,t-1): 0.9768435716629028
stopping after 6 steps
covariate-adjusted NAM not found; computing and saving
computing SVD
performing association test


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


computing neighborhood-level FDRs
1
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.3812618738126187 , 1 PCs used
total r^2 between top 1 NAM PCs and outcome is 0.02


In [110]:
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/chronicity/sc_ncorr.csv", 
               res.ncorrs, delimiter=",")
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/chronicity/sc_fdrs.csv", 
               res.fdrs, delimiter=",")

In [111]:
sc_uni['Activity'] = res.p

# ISN

## Single Cell

In [112]:
meta = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/ISN/sc_meta.csv')
print(meta.shape)

(35187, 39)


In [113]:
harmony = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/ISN/sc_harmony.csv')
print(harmony.shape)

(35187, 20)


In [114]:
umap = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/ISN/sc_umap.csv')
print(umap.shape)

(35187, 2)


In [115]:
# Start by making anndata object
d = mad.MultiAnnData(X=harmony, obs=meta, sampleid="sample")
#d.obs_to_sample(['Type', 'Site',
#                 'Responder_Status', 'Sex',
#                 'Age', 'Race',
#                'Ethnicity', 'ISN',
#               'Activity', "Chronicity",
#                 'processing_batch'])
d.obs_to_sample(['Final_ISN_[III]', 'Final_ISN_[III][V]', 'Final_ISN_[IV]',
                 'Final_ISN_[IV][V]', 'Final_ISN_[V]'])
d.samplem.head()

['cell' 'sample' 'final_annotation' 'Responder.Status' 'Race' 'Final_ISN'
 'Type' 'Final_Site']
consider casting to numeric types where appropriate, and
consider re-coding text-valued columns with pandas.get_dummies


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/multianndata/core.py:17: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  super().__init__(*args, **kwargs)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


,Final_ISN_[III],Final_ISN_[III][V],Final_ISN_[IV],Final_ISN_[IV][V],Final_ISN_[V]
sample,,,,,
AMPSLEkid_cells_0134,1.0,0.0,0.0,0.0,0.0
AMPSLEkid_cells_0137,1.0,0.0,0.0,0.0,0.0
AMPSLEkid_cells_0138,0.0,1.0,0.0,0.0,0.0
AMPSLEkid_cells_0139,0.0,1.0,0.0,0.0,0.0
AMPSLEkid_cells_0140,0.0,0.0,0.0,0.0,1.0


In [116]:
umap.index = d.obs.index
d.obsm['X_umap'] = umap

In [117]:
np.random.seed(0) 
cna.pp.knn(d)

computing default knn graph


In [118]:
np.random.seed(0) 
res = cna.tl._association.association(d, #dataset 
                                      d.samplem['Final_ISN_[III]'], #phenotype
#                                       batches=d.samplem.processing_batch, #batches
                                      Nnull=10000, # number of null permutations to do (defaults to only 1e3)
                                      ks=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] # I asked the method to consider up to 10 PCs because
                                                                #it chose the max number of PCs it considered the default set of [1,2,3,4]
                                     )
print(res.k)
print(res.ks)
print('p =', res.p, ',', res.k, 'PCs used')
print('total r^2 between top {} NAM PCs and outcome is {:.2f}'.format(res.k, res.r2))

qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 52.71991729736328
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 33.68928527832031
	20th percentile R2(t,t-1): 0.7436593174934387
	taking step 3
	median kurtosis: 23.012622833251953
	20th percentile R2(t,t-1): 0.8982551097869873
	taking step 4
	median kurtosis: 16.160520553588867
	20th percentile R2(t,t-1): 0.93894362449646
	taking step 5
	median kurtosis: 12.253840446472168
	20th percentile R2(t,t-1): 0.9611961245536804
	taking step 6
	median kurtosis: 10.029513359069824
	20th percentile R2(t,t-1): 0.976091206073761
stopping after 6 steps
covariate-adjusted NAM not found; computing and saving
computing SVD


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


performing association test
computing neighborhood-level FDRs
19
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.7706229377062294 , 19 PCs used
total r^2 between top 19 NAM PCs and outcome is 0.13


In [119]:
np.random.seed(0) 
res = cna.tl._association.association(d, #dataset 
                                      d.samplem['Final_ISN_[III][V]'], #phenotype
#                                       batches=d.samplem.processing_batch, #batches
                                      Nnull=10000, # number of null permutations to do (defaults to only 1e3)
                                      ks=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] # I asked the method to consider up to 10 PCs because
                                                                #it chose the max number of PCs it considered the default set of [1,2,3,4]
                                     )
print(res.k)
print(res.ks)
print('p =', res.p, ',', res.k, 'PCs used')
print('total r^2 between top {} NAM PCs and outcome is {:.2f}'.format(res.k, res.r2))

performing association test
computing neighborhood-level FDRs
5
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.8034196580341966 , 5 PCs used
total r^2 between top 5 NAM PCs and outcome is 0.04


In [121]:
np.random.seed(0) 
res = cna.tl._association.association(d, #dataset 
                                      d.samplem['Final_ISN_[IV]'], #phenotype
#                                       batches=d.samplem.processing_batch, #batches
                                      Nnull=10000, # number of null permutations to do (defaults to only 1e3)
                                      ks=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] # I asked the method to consider up to 10 PCs because
                                                                #it chose the max number of PCs it considered the default set of [1,2,3,4]
                                     )
print(res.k)
print(res.ks)
print('p =', res.p, ',', res.k, 'PCs used')
print('total r^2 between top {} NAM PCs and outcome is {:.2f}'.format(res.k, res.r2))

performing association test
computing neighborhood-level FDRs
6
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.32756724327567244 , 6 PCs used
total r^2 between top 6 NAM PCs and outcome is 0.07


In [122]:
np.random.seed(0) 
res = cna.tl._association.association(d, #dataset 
                                      d.samplem['Final_ISN_[IV][V]'], #phenotype
#                                       batches=d.samplem.processing_batch, #batches
                                      Nnull=10000, # number of null permutations to do (defaults to only 1e3)
                                      ks=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] # I asked the method to consider up to 10 PCs because
                                                                #it chose the max number of PCs it considered the default set of [1,2,3,4]
                                     )
print(res.k)
print(res.ks)
print('p =', res.p, ',', res.k, 'PCs used')
print('total r^2 between top {} NAM PCs and outcome is {:.2f}'.format(res.k, res.r2))

performing association test
computing neighborhood-level FDRs
5
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.33366663333666635 , 5 PCs used
total r^2 between top 5 NAM PCs and outcome is 0.07


In [123]:
np.random.seed(0) 
res = cna.tl._association.association(d, #dataset 
                                      d.samplem['Final_ISN_[V]'], #phenotype
#                                       batches=d.samplem.processing_batch, #batches
                                      Nnull=10000, # number of null permutations to do (defaults to only 1e3)
                                      ks=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] # I asked the method to consider up to 10 PCs because
                                                                #it chose the max number of PCs it considered the default set of [1,2,3,4]
                                     )
print(res.k)
print(res.ks)
print('p =', res.p, ',', res.k, 'PCs used')
print('total r^2 between top {} NAM PCs and outcome is {:.2f}'.format(res.k, res.r2))

performing association test
computing neighborhood-level FDRs
1
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.24777522247775222 , 1 PCs used
total r^2 between top 1 NAM PCs and outcome is 0.03


In [124]:
sc_uni['ISN'] = res.p

In [182]:
pd.DataFrame.from_dict(sc_uni, orient='index').T.to_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/univariate_cna.csv')

# INJURED PT

## Single Cell

In [5]:
meta = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/injured_pt/sc_meta.csv')
print(meta.shape)

(34203, 40)


In [6]:
harmony = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/injured_pt/sc_harmony.csv')
print(harmony.shape)

(34203, 20)


In [7]:
umap = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/injured_pt/sc_umap.csv')
print(umap.shape)

(34203, 2)


In [8]:
res = cna_test(meta, harmony, umap, 'injured_pt_prop', covars = ['Final_Site_JHU',
                                                                  'Final_Site_NYU',
                                                                  'First_biop',
                                                                  'Responder_Status',
                                                                  'Age'])

(34203, 40)
(34203, 20)
(34203, 2)
['cell' 'sample' 'final_annotation' 'Responder.Status' 'Race' 'Final_ISN'
 'Type' 'Final_Site']
consider casting to numeric types where appropriate, and
consider re-coding text-valued columns with pandas.get_dummies
computing default knn graph


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/multianndata/core.py:17: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  super().__init__(*args, **kwargs)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 52.52780532836914
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 33.63969802856445
	20th percentile R2(t,t-1): 0.7439629435539246
	taking step 3
	median kurtosis: 23.036291122436523
	20th percentile R2(t,t-1): 0.8980184197425842
	taking step 4
	median kurtosis: 16.27391242980957
	20th percentile R2(t,t-1): 0.9390338659286499
	taking step 5
	median kurtosis: 12.294208526611328
	20th percentile R2(t,t-1): 0.9612075686454773
	taking step 6
	median kurtosis: 10.025031089782715
	20th percentile R2(t,t-1): 0.9760475158691406
stopping after 6 steps
covariate-adjusted NAM not found; computing and saving
computing SVD


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


performing association test
computing neighborhood-level FDRs
14
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.0025997400259974 , 14 PCs used
total r^2 between top 14 NAM PCs and outcome is 0.30


In [9]:
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/injured_pt/sc_ncorr_cond_nochron.csv", 
               res.ncorrs, delimiter=",")
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/injured_pt/sc_fdrs_cond_nochron.csv", 
               res.fdrs, delimiter=",")

In [6]:
res = cna_test(meta, harmony, umap, 'injured_pt_prop', covars = ['Final_Site_JHU',
                                                                  'Final_Site_NYU',
                                                                  'First_biop',
                                                                  'Responder_Status',
                                                                  'Age',
                                                                  'Final_Chronicity'])

(34203, 40)
(34203, 20)
(34203, 2)
['cell' 'sample' 'final_annotation' 'Responder.Status' 'Race' 'Final_ISN'
 'Type' 'Final_Site']
consider casting to numeric types where appropriate, and
consider re-coding text-valued columns with pandas.get_dummies
computing default knn graph


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/multianndata/core.py:17: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  super().__init__(*args, **kwargs)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 52.52780532836914
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 33.63969802856445
	20th percentile R2(t,t-1): 0.7439629435539246
	taking step 3
	median kurtosis: 23.036291122436523
	20th percentile R2(t,t-1): 0.8980184197425842
	taking step 4
	median kurtosis: 16.27391242980957
	20th percentile R2(t,t-1): 0.9390338659286499
	taking step 5
	median kurtosis: 12.294208526611328
	20th percentile R2(t,t-1): 0.9612075686454773
	taking step 6
	median kurtosis: 10.025031089782715
	20th percentile R2(t,t-1): 0.9760475158691406
stopping after 6 steps
covariate-adjusted NAM not found; computing and saving
computing SVD
performing association test


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


computing neighborhood-level FDRs
3
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.9989001099890011 , 3 PCs used
total r^2 between top 3 NAM PCs and outcome is 0.01


In [11]:
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/injured_pt/sc_cond_ncorr.csv", 
               res.ncorrs, delimiter=",")
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/t_nk/injured_pt/sc_cond_fdrs.csv", 
               res.fdrs, delimiter=",")